# Synthetic Data Generation for Food Waste Records with Timestamps
### Objective
This notebook augments the existing food waste dataset by creating synthetic rows that statistically resemble the original data. The synthetic data preserves the distributions, relationships, and patterns observed in the real records. **Crucially, each record now receives a realistic time of day based on meal periods**, enabling later aggregation to 30‑minute intervals with meaningful patterns. The final output is a single CSV file containing both the original and the newly generated data with full timestamps.

## 1. Setup and Data Loading
We begin by importing the necessary libraries and loading the original dataset. We set a random seed to ensure reproducibility of the synthetic generation process.

In [26]:
import os
import pandas as pd
import numpy as np
from datetime import timedelta

# Set seed for reproducibility
np.random.seed(42)

In [27]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Change to your own directory
try:
    os.chdir("/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence")
    print("Directory changed")
except OSError:
    print("Error: Can't change the Current Working Directory")

Mounted at /content/drive
Directory changed


In [28]:
# Load the original data
df_original = pd.read_csv('data/food_waste_cleaned.csv')
print(f"Original data shape: {df_original.shape}")

Original data shape: (2600, 23)


## 2. Add Realistic Times to Original Records
The original data has dates only. We now assign an hour and minute based on the recorded `Meal` to create realistic meal‑time clusters.

In [29]:
# Define meal time windows
meal_time_windows = {
    "Breakfast": (6, 9),
    "Lunch": (11, 14),
    "Dinner": (17, 20)
}

def meal_based_time(meal, date):
    """Return a datetime with hour inside the meal window."""
    start, end = meal_time_windows[meal]
    hour = np.random.randint(start, end)
    minute = np.random.randint(0, 60)
    return date + pd.Timedelta(hours=hour, minutes=minute)

# Convert Date column to datetime
df_original['Date'] = pd.to_datetime(df_original['Date'])

# Apply meal‑based times
df_original['Date'] = df_original.apply(lambda row: meal_based_time(row['Meal'], row['Date']), axis=1)

print("First few dates after adding realistic times:")
print(df_original['Date'].head())

First few dates after adding realistic times:
0   2025-06-11 08:51:00
1   2025-06-11 11:14:00
2   2025-06-11 19:07:00
3   2025-06-11 11:20:00
4   2025-06-11 19:57:00
Name: Date, dtype: datetime64[ns]


## 3. Understanding the Data
Before generating synthetic data, we examine the statistical properties of each column. We separate columns into categorical, numerical, and derived ones. Some columns (like `Cost_Loss`) are fully determined by others and will be recomputed after generation.

In [30]:
# Display basic info
df_original.info()
df_original.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2600 entries, 0 to 2599
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               2600 non-null   datetime64[ns]
 1   Meal               2600 non-null   object        
 2   Canteen_Section    2600 non-null   object        
 3   Food_Category      2600 non-null   object        
 4   Waste_Weight_kg    2600 non-null   float64       
 5   Unit_Price_per_kg  2600 non-null   float64       
 6   Cost_Loss          2600 non-null   float64       
 7   Year               2600 non-null   int64         
 8   Month              2600 non-null   int64         
 9   Day                2600 non-null   int64         
 10  Weekday            2600 non-null   object        
 11  Week               2600 non-null   int64         
 12  DayOfYear          2600 non-null   int64         
 13  Quarter            2600 non-null   int64         
 14  IsWeeken

,Date,Meal,Canteen_Section,Food_Category,Waste_Weight_kg,Unit_Price_per_kg,Cost_Loss,Year,Month,Day,...,Quarter,IsWeekend,Is_Waste_Outlier,Is_Cost_Outlier,Is_High_Waste,Is_High_Cost,Month_Name,Weekday_Type,Waste_per_Price,Log_Waste
0,2025-06-11 08:51:00,Breakfast,B,Vegetables,1.81,3.0,5.430,2025,6,11,...,2,False,False,False,False,False,June,Weekday,0.603333,1.033184
1,2025-06-11 11:14:00,Lunch,B,Soup,3.32,1.5,4.980,2025,6,11,...,2,False,False,False,False,False,June,Weekday,2.213333,1.463255
2,2025-06-11 19:07:00,Dinner,B,Soup,1.27,1.5,1.905,2025,6,11,...,2,False,False,False,False,False,June,Weekday,0.846667,0.819780
3,2025-06-11 11:20:00,Lunch,D,Soup,1.10,1.5,1.650,2025,6,11,...,2,False,False,False,False,False,June,Weekday,0.733333,0.741937
4,2025-06-11 19:57:00,Dinner,D,Meat,4.57,8.0,36.560,2025,6,11,...,2,False,False,True,True,True,June,Weekday,0.571250,1.717395


### 3.1 Categorical Variables
The main categorical variables are: `Meal`, `Canteen_Section`, `Food_Category`. Their distributions will be used to sample new values.

In [31]:
# Frequency tables for categorical columns
meal_dist = df_original['Meal'].value_counts(normalize=True)
section_dist = df_original['Canteen_Section'].value_counts(normalize=True)
category_dist = df_original['Food_Category'].value_counts(normalize=True)

print("Meal distribution:\n", meal_dist)
print("\nCanteen Section distribution:\n", section_dist)
print("\nFood Category distribution:\n", category_dist)

Meal distribution:
 Meal
Breakfast    0.334231
Lunch        0.333462
Dinner       0.332308
Name: proportion, dtype: float64

Canteen Section distribution:
 Canteen_Section
B    0.251538
A    0.250769
D    0.249231
C    0.248462
Name: proportion, dtype: float64

Food Category distribution:
 Food_Category
Rice          0.253462
Meat          0.251154
Vegetables    0.250385
Soup          0.245000
Name: proportion, dtype: float64


### 3.2 Numerical Variables
The key numerical variable is `Waste_Weight_kg`. `Unit_Price_per_kg` is constant per food category (Meat = 8.0, Vegetables = 3.0, Soup = 1.5, Rice = 2.0). Therefore, we model `Waste_Weight_kg` separately for each food category, because the price is fixed and does not vary within a category.

In [32]:
# Check Unit_Price per Food_Category
price_map_check = df_original.groupby('Food_Category')['Unit_Price_per_kg'].unique()
print(price_map_check)  # Confirm each category has a single price

Food_Category
Meat          [8.0]
Rice          [2.0]
Soup          [1.5]
Vegetables    [3.0]
Name: Unit_Price_per_kg, dtype: object


In [33]:
# Compute log of Waste_Weight (natural log) for distribution fitting
df_original['Log_Waste_check'] = np.log(df_original['Waste_Weight_kg'])

category_params = {}
for cat in df_original['Food_Category'].unique():
    subset = df_original[df_original['Food_Category'] == cat]['Log_Waste_check']
    mean_log = subset.mean()
    std_log = subset.std()
    category_params[cat] = {'mean_log': mean_log, 'std_log': std_log}
    print(f"{cat}: log mean = {mean_log:.3f}, log std = {std_log:.3f}")

Vegetables: log mean = 0.721, log std = 0.802
Soup: log mean = 0.756, log std = 0.787
Meat: log mean = 0.643, log std = 0.851
Rice: log mean = 0.730, log std = 0.793


### 3.3 Date Variables
The original data now spans from `{df_original['Date'].min()}` to `{df_original['Date'].max()}`. We will generate new timestamps within a similar range but ensure they maintain the empirical day‑of‑week distribution.

In [34]:
min_date = df_original['Date'].min()
max_date = df_original['Date'].max()
date_range = (max_date - min_date).days
print(f"Date range: {min_date} to {max_date} ({date_range} days)")

# Extract day of week distribution (Monday=0, Sunday=6)
dow_dist = df_original['Date'].dt.dayofweek.value_counts(normalize=True).sort_index()
print("\nDay-of-week distribution (0=Monday):\n", dow_dist)

Date range: 2025-06-11 06:01:00 to 2025-08-10 19:59:00 (60 days)

Day-of-week distribution (0=Monday):
 Date
0    0.132692
1    0.130769
2    0.145769
3    0.148846
4    0.146154
5    0.145000
6    0.150769
Name: proportion, dtype: float64


### 3.4 Flag Columns
The flags (`Is_Waste_Outlier`, `Is_Cost_Outlier`, `Is_High_Waste`, `Is_High_Cost`) are based on thresholds. We compute these thresholds from the original data and apply the same rules to the synthetic data.

In [35]:
# Compute thresholds from original data
q1_waste = df_original['Waste_Weight_kg'].quantile(0.25)
q3_waste = df_original['Waste_Weight_kg'].quantile(0.75)
iqr_waste = q3_waste - q1_waste
waste_lower_bound = q1_waste - 1.5 * iqr_waste
waste_upper_bound = q3_waste + 1.5 * iqr_waste

q1_cost = df_original['Cost_Loss'].quantile(0.25)
q3_cost = df_original['Cost_Loss'].quantile(0.75)
iqr_cost = q3_cost - q1_cost
cost_lower_bound = q1_cost - 1.5 * iqr_cost
cost_upper_bound = q3_cost + 1.5 * iqr_cost

high_waste_thresh = df_original['Waste_Weight_kg'].quantile(0.90)
high_cost_thresh = df_original['Cost_Loss'].quantile(0.90)

print(f"Waste outlier bounds: [{waste_lower_bound:.2f}, {waste_upper_bound:.2f}]")
print(f"Cost outlier bounds: [{cost_lower_bound:.2f}, {cost_upper_bound:.2f}]")
print(f"High waste threshold (90th percentile): {high_waste_thresh:.2f}")
print(f"High cost threshold (90th percentile): {high_cost_thresh:.2f}")

Waste outlier bounds: [-2.23, 7.42]
Cost outlier bounds: [-7.69, 22.13]
High waste threshold (90th percentile): 4.52
High cost threshold (90th percentile): 23.77


## 4. Generating Synthetic Data with Realistic Temporal Patterns
We will generate synthetic rows by creating **event batches** that mimic real operational patterns:
* Waste happens around meal times.
* Multiple waste events occur within the same 30‑minute window.
* Some windows have no waste (controlled by `ZERO_WINDOW_PROB`).
* Waste weight depends on meal and section (multipliers).

In [36]:
# Fixed price mapping
price_map = {'Meat': 8.0, 'Vegetables': 3.0, 'Soup': 1.5, 'Rice': 2.0}

# Multipliers for waste weight based on meal and section
meal_multiplier = {
    "Breakfast": 0.8,
    "Lunch": 1.3,
    "Dinner": 1.1
}

section_multiplier = {
    "A": 1.0,
    "B": 1.1,
    "C": 0.9,
    "D": 1.2
}

# Zero-window probability
ZERO_WINDOW_PROB = 0.25

# Target total rows: 120,000. We'll generate in batches (windows) with 2–6 events per window.
TARGET_ROWS = 1000000
AVG_EVENTS_PER_WINDOW = 4
N_WINDOWS = TARGET_ROWS // AVG_EVENTS_PER_WINDOW  # 30000

# Helper function to generate a random date (preserving day‑of‑week distribution)
def generate_base_date():
    dow = np.random.choice(dow_dist.index, p=dow_dist.values)
    random_days = np.random.randint(0, date_range + 1)
    base_date = min_date + timedelta(days=int(random_days))
    days_ahead = int(dow) - base_date.weekday()
    if days_ahead > 0:
        candidate = base_date + timedelta(days=days_ahead)
    else:
        candidate = base_date + timedelta(days=days_ahead + 7)
    if candidate > max_date:
        candidate -= timedelta(weeks=1)
    shift = np.random.randint(-3, 4)
    final_date = candidate + timedelta(days=int(shift))
    if final_date < min_date:
        final_date = min_date
    if final_date > max_date:
        final_date = max_date
    return final_date

synthetic_rows = []

for _ in range(N_WINDOWS):
    # 1. Sample categoricals for this window
    meal = np.random.choice(meal_dist.index, p=meal_dist.values)
    section = np.random.choice(section_dist.index, p=section_dist.values)
    category = np.random.choice(category_dist.index, p=category_dist.values)

    # 2. Generate base datetime: date + hour from meal window
    base_date = generate_base_date()
    start_hour, end_hour = meal_time_windows[meal]
    hour = np.random.randint(start_hour, end_hour)
    # Ensure minute aligns to 30‑minute bucket (0 or 30) – cast to int for timedelta
    bucket_minute = int(np.random.choice([0, 30]))
    base_datetime = base_date + timedelta(hours=int(hour), minutes=bucket_minute)

    # 3. Decide if this window has waste
    is_zero_window = np.random.rand() < ZERO_WINDOW_PROB

    # 4. Generate multiple events within this window
    events_this_window = np.random.randint(2, 7)  # 2–6 events
    for j in range(events_this_window):
        # Minute offset between 0 and 29, so all events fall in same 30‑min bucket
        minute_offset = int(np.random.randint(0, 30))
        timestamp = base_datetime + timedelta(minutes=minute_offset)

        if is_zero_window:
            waste_weight = 0.0
        else:
            # Base waste from log‑normal for this category
            params = category_params[category]
            log_waste = np.random.normal(params['mean_log'], params['std_log'])
            base_waste = np.exp(log_waste)
            # Apply multipliers
            waste_weight = base_waste * meal_multiplier[meal] * section_multiplier[section]
            waste_weight = abs(waste_weight)  # ensure positive

        unit_price = price_map[category]
        cost_loss = waste_weight * unit_price
        waste_per_price = waste_weight / unit_price if unit_price != 0 else 0
        log_waste_calc = np.log(waste_weight) if waste_weight > 0 else 0

        # Date-time features
        year = timestamp.year
        month = timestamp.month
        day = timestamp.day
        hour = timestamp.hour
        minute = timestamp.minute
        weekday_num = timestamp.weekday()
        weekday_name = timestamp.strftime('%A')
        week = timestamp.isocalendar()[1]
        day_of_year = timestamp.timetuple().tm_yday
        quarter = (month - 1) // 3 + 1
        is_weekend = 1 if weekday_num >= 5 else 0
        month_name = timestamp.strftime('%B')
        weekday_type = 'Weekend' if is_weekend else 'Weekday'

        # Flags
        is_waste_outlier = 1 if (waste_weight < waste_lower_bound or waste_weight > waste_upper_bound) else 0
        is_cost_outlier = 1 if (cost_loss < cost_lower_bound or cost_loss > cost_upper_bound) else 0
        is_high_waste = 1 if waste_weight > high_waste_thresh else 0
        is_high_cost = 1 if cost_loss > high_cost_thresh else 0

        row = {
            'Date': timestamp,
            'Meal': meal,
            'Canteen_Section': section,
            'Food_Category': category,
            'Waste_Weight_kg': round(waste_weight, 2),
            'Unit_Price_per_kg': unit_price,
            'Cost_Loss': round(cost_loss, 2),
            'Year': year,
            'Month': month,
            'Day': day,
            'Hour': hour,
            'Minute': minute,
            'Weekday': weekday_name,
            'Week': week,
            'DayOfYear': day_of_year,
            'Quarter': quarter,
            'IsWeekend': is_weekend,
            'Is_Waste_Outlier': is_waste_outlier,
            'Is_Cost_Outlier': is_cost_outlier,
            'Is_High_Waste': is_high_waste,
            'Is_High_Cost': is_high_cost,
            'Month_Name': month_name,
            'Weekday_Type': weekday_type,
            'Waste_per_Price': round(waste_per_price, 6),
            'Log_Waste': round(log_waste_calc, 6)
        }
        synthetic_rows.append(row)

df_synthetic = pd.DataFrame(synthetic_rows)
print(f"Synthetic data shape: {df_synthetic.shape}")

Synthetic data shape: (1000014, 25)


## 5. Combine Original and Synthetic Data
Now we concatenate the original dataframe (with its new timestamps) with the synthetic data.

In [37]:
# Drop temporary column if it exists
if 'Log_Waste_check' in df_original.columns:
    df_original = df_original.drop(columns=['Log_Waste_check'])

# Combine
df_combined = pd.concat([df_original, df_synthetic], ignore_index=True)
print(f"Combined data shape: {df_combined.shape}")
df_combined.head()

Combined data shape: (1002614, 25)


,Date,Meal,Canteen_Section,Food_Category,Waste_Weight_kg,Unit_Price_per_kg,Cost_Loss,Year,Month,Day,...,Is_Waste_Outlier,Is_Cost_Outlier,Is_High_Waste,Is_High_Cost,Month_Name,Weekday_Type,Waste_per_Price,Log_Waste,Hour,Minute
0,2025-06-11 08:51:00,Breakfast,B,Vegetables,1.81,3.0,5.430,2025,6,11,...,0,0,0,0,June,Weekday,0.603333,1.033184,NaN,NaN
1,2025-06-11 11:14:00,Lunch,B,Soup,3.32,1.5,4.980,2025,6,11,...,0,0,0,0,June,Weekday,2.213333,1.463255,NaN,NaN
2,2025-06-11 19:07:00,Dinner,B,Soup,1.27,1.5,1.905,2025,6,11,...,0,0,0,0,June,Weekday,0.846667,0.819780,NaN,NaN
3,2025-06-11 11:20:00,Lunch,D,Soup,1.10,1.5,1.650,2025,6,11,...,0,0,0,0,June,Weekday,0.733333,0.741937,NaN,NaN
4,2025-06-11 19:57:00,Dinner,D,Meat,4.57,8.0,36.560,2025,6,11,...,0,1,1,1,June,Weekday,0.571250,1.717395,NaN,NaN


## 6. Save to CSV
Finally, we save the combined dataset to a new CSV file. Pandas will automatically format the datetime column in ISO 8601 format (e.g., `2025-06-11 08:23:00`).

In [38]:
output_filename = 'data/food_waste_augmented.csv'
df_combined.to_csv(output_filename, index=False)
print(f"File saved as {output_filename}")

File saved as data/food_waste_augmented.csv


## 7. Verify 30‑Minute Aggregation Density
We resample the combined data to 30‑minute intervals and check the ratio of zero vs. non‑zero windows. A healthy dataset for time‑series forecasting should have 20–35% zero windows.

In [39]:
# Ensure datetime index
df_combined = df_combined.sort_values("Date")
agg = (
    df_combined.set_index("Date")
    .resample("30min")["Waste_Weight_kg"]
    .sum()
)

total_windows = len(agg)
zero_windows = (agg == 0).sum()
nonzero_windows = (agg > 0).sum()

print(f"Total 30‑min windows: {total_windows}")
print(f"Zero windows: {zero_windows} ({zero_windows/total_windows*100:.1f}%)")
print(f"Non‑zero windows: {nonzero_windows} ({nonzero_windows/total_windows*100:.1f}%)")

Total 30‑min windows: 2948
Zero windows: 1200 (40.7%)
Non‑zero windows: 1748 (59.3%)


## 8. Conclusion
The augmented dataset now contains a large number of records **with realistic timestamps** and meaningful temporal patterns (meal clusters, multiple events per window, controlled zeros). This enables time‑series modeling at 30‑minute resolution, providing a rich training set for forecasting models.